In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
with open('Anna.txt', 'r') as file:
    text = file.read()

In [3]:
chars = tuple(set(text))
int_char = dict(enumerate(chars))
char_int = {ch : i for i, ch in int_char.items()}

encoded = np.array([char_int[x] for x in text])
encoded[:15]

array([48, 15,  2, 19, 58, 18, 25, 45, 78, 39, 39, 39, 61,  2, 19])

In [4]:
def one_hot_encode(arr, label):
    one_hot = np.zeros((arr.size, label), dtype=np.float32)
    one_hot[np.arange(one_hot.shape[0]), arr.flatten()] = 1
    one_hot = one_hot.reshape((*arr.shape,label))
    
    return one_hot

In [5]:
def get_batches(arr, batch_size, seq_len):
    total_seq = batch_size*seq_len
    n_batch = len(arr)//(total_seq)
    arr = arr[:n_batch*(total_seq)]
    arr = arr.reshape((batch_size,-1))

    for n in range(0, arr.shape[1], seq_len):
        x = arr[:, n:n+seq_len]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_len]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [6]:
batches = get_batches(encoded, 8, 50)
x, y = next(batches)

In [7]:
print('x\n', x[:10, :10])
print('y\n', y[:10, :10])

x
 [[48 15  2 19 58 18 25 45 78 39]
 [35  4 16 45 58 15  2 58 45  2]
 [18 16 60 45  4 25 45  2 45 77]
 [35 45 58 15 18 45 81 15  0 18]
 [45 35  2 42 45 15 18 25 45 58]
 [81 70 35 35  0  4 16 45  2 16]
 [45 65 16 16  2 45 15  2 60 45]
 [ 3 64 59  4 16 35 36 12 29 45]]
y
 [[15  2 19 58 18 25 45 78 39 39]
 [ 4 16 45 58 15  2 58 45  2 58]
 [16 60 45  4 25 45  2 45 77  4]
 [45 58 15 18 45 81 15  0 18 77]
 [35  2 42 45 15 18 25 45 58 18]
 [70 35 35  0  4 16 45  2 16 60]
 [65 16 16  2 45 15  2 60 45 35]
 [64 59  4 16 35 36 12 29 45 46]]


In [8]:
class CharNN(nn.Module):    
    def __init__(self, tokens, n_layer=2, n_hidden=256, drop_prob=0.5, lr=0.001):
        super().__init__()

        self.n_layer = n_layer
        self.n_hidden = n_hidden
        self.lr = lr

        self.chars = tokens
        self.int_char = dict(enumerate(self.chars))
        self.char_int = {ch : i for i, ch in self.int_char.items()}

        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layer, dropout=drop_prob, batch_first=True)

        self.dropout = nn.Dropout(drop_prob)

        self.fc = nn.Linear(n_hidden, len(self.chars))

    def forward(self, x, hidden):
        r_output, hidden = self.lstm(x, hidden)

        out = self.dropout(r_output)
       
        out = out.contiguous().view(-1, self.n_hidden)

        out = self.fc(out)

        return out, hidden
    
    def init_hidden (self, batch_size):
        
        weight = next(self.parameters()).data
        
        hidden = (weight.new(self.n_layer, batch_size, self.n_hidden).zero_(),
                  weight.new(self.n_layer, batch_size, self.n_hidden).zero_())
        
        return hidden

In [9]:
def train(net, data, epoch=10, batch_size=10, seq_len=50, lr=0.001, clip=5, val_frac=0.1):

    net.train()

    opt = torch.optim.Adam(net.parameters(), lr=lr)
    crietrion = nn.CrossEntropyLoss()

    val_idx = int(len(data)*(1-val_frac))
    data, val_data = data[:val_idx], data[val_idx:]

    n_chars = len(net.chars)

    for n in range(epoch):
        h = net.init_hidden(batch_size)

        for x, y in get_batches(data, batch_size, seq_len):
            x = one_hot_encode(x, n_chars)
            input, target = torch.from_numpy(x), torch.from_numpy(y)

            h = tuple([each.data for each in h])

            net.zero_grad()
            output, h = net(input, h)
            loss = crietrion(output, target.view(batch_size*seq_len).long())

            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
        
        else:
            val_losses = []
            net.eval()
            val_h = net.init_hidden(batch_size)

            for x, y in get_batches(val_data, batch_size, seq_len):
                x = one_hot_encode(x,n_chars)
                inputs, targets = torch.from_numpy(x), torch.from_numpy(y)

                val_h = tuple([each.data for each in val_h])

                output, val_h = net(inputs, val_h)
                val_loss = crietrion(output, targets.view(batch_size*seq_len).long())

                val_losses.append(val_loss.item())

            net.train()

            print(f"Epoch: {n+1}, Train loss: {loss.item():.4f}, Val loss: {np.mean(val_losses):.4f}")

In [10]:
n_hidden = 512
n_layer = 2

net = CharNN(chars, n_layer=n_layer, n_hidden=n_hidden)
print(net)

CharNN(
  (lstm): LSTM(83, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=83, bias=True)
)


In [11]:
batch_size = 128
seq_len = 100
epoch = 20

train(net, encoded, epoch, batch_size, seq_len)

Epoch: 1, Train loss: 2.6448, Val loss: 2.6251
Epoch: 2, Train loss: 2.1692, Val loss: 2.1362
Epoch: 3, Train loss: 1.9264, Val loss: 1.8933
Epoch: 4, Train loss: 1.7792, Val loss: 1.7330
Epoch: 5, Train loss: 1.6742, Val loss: 1.6280
Epoch: 6, Train loss: 1.5912, Val loss: 1.5626
Epoch: 7, Train loss: 1.5384, Val loss: 1.5039
Epoch: 8, Train loss: 1.4940, Val loss: 1.4608
Epoch: 9, Train loss: 1.4433, Val loss: 1.4256
Epoch: 10, Train loss: 1.4115, Val loss: 1.3962
Epoch: 11, Train loss: 1.3841, Val loss: 1.3757
Epoch: 12, Train loss: 1.3573, Val loss: 1.3542
Epoch: 13, Train loss: 1.3303, Val loss: 1.3295
Epoch: 14, Train loss: 1.3157, Val loss: 1.3210
Epoch: 15, Train loss: 1.2994, Val loss: 1.3153
Epoch: 16, Train loss: 1.2949, Val loss: 1.2964
Epoch: 17, Train loss: 1.2736, Val loss: 1.2900
Epoch: 18, Train loss: 1.2545, Val loss: 1.2881
Epoch: 19, Train loss: 1.2495, Val loss: 1.2757
Epoch: 20, Train loss: 1.2398, Val loss: 1.2710


In [12]:
model_name = 'CheckPoint_20.pth'

checkpoint = {
    'n_hidden': net.n_hidden,
    'n_layer': net.n_layer,
    'state_dict': net.state_dict(),
    'chars': net.chars
}

with open(model_name, 'wb') as f:
    torch.save(checkpoint, f)

In [13]:
with open('CheckPoint_20.pth', 'rb') as f:
	checkpoint = torch.load(f)

loaded = CharNN(tokens=checkpoint['chars'], n_hidden=checkpoint['n_hidden'], n_layer=checkpoint['n_layer'])
loaded.load_state_dict(checkpoint['state_dict'])
print(loaded)

C:\Users\Yonatan\AppData\Local\Temp\ipykernel_7420\3318694581.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f)


CharNN(
  (lstm): LSTM(83, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=83, bias=True)
)


In [14]:
def Predict(net, char, h=None, top_k=None):
    x = np.array([[net.char_int[char]]])
    x = one_hot_encode(x, len(net.chars))
    inputs = torch.from_numpy(x)

    h = tuple([one for one in h])

    out, h = net(inputs, h)
    p = F.softmax(out, dim=1).data

    if top_k is None:
        top_ch = np.arange(len(net.chars))
    else:
        p, top_ch = p.topk(top_k)
        top_ch = top_ch.numpy().squeeze()

    p = p.numpy().squeeze()
    char = np.random.choice(top_ch, p=p/p.sum())

    return net.int_char[char], h

In [15]:
def Sample(net, size, prime='The',top_k=None):
    
    net.eval()

    chars = [ch for ch in prime]
    h = net.init_hidden(1)

    for ch in chars:
        char, h = Predict(net, ch, h, top_k=top_k)
    
    chars.append(char)

    for n in range(size):
        char, h = Predict(net, chars[-1], h, top_k=top_k)
        chars.append(char)

    return ''.join(chars)

In [16]:
print(Sample(loaded, 1000, 'The', top_k=5))

Ther, and she was
stord in a carriage in the country, and he carried away, the sates of tee or
the plans where he had so much, and she was anguily asking a minute,
than the chashing had been so fan in the station of the condression where
she danced the candle of his belonger, and went as all of his being
for the cartion, and straight a sudden and still soul and sat something on
the same chird about her at once told, but with what had seen a secret
composure with him. The mustress that she had saying the princess, he
had to charte over it.

"That wouldn't tell me that he shouldn't be stretching her. They say
to her....

"Well, then all the same, if you don't come to the thoughts of the
marsh would have been the country. Better now you can be so like anyway. Then I
say it's so taken an hour of an interview of my pleasures. I have better
train..."

"I am the fact, and the prince were a little, and what it's as a support of
husting," she said to himself... After so string how he sent the c